# 1.3. Thu thập Trích dẫn Semantic Scholar và Độ phủ Metadata

## 1. Kết hợp Batch API, Phân trang kiểm chứng `referenceCount` và Bảo toàn ID trước 2007

- **Truy vấn lô trường phẳng (`POST /graph/v1/paper/batch`)**: Truy vấn tối đa `50` bài/lô (nhịp `5,0s`, tự chia đôi đệ quy nếu phản hồi vượt `10 MB` / HTTP 413) với danh sách trường phẳng chuẩn (`externalIds,referenceCount,references.paperId,references.title,references.authors,references.year,references.publicationDate,references.venue,references.externalIds`) và đối chiếu ngược `externalIds.ArXiv` để chống ánh xạ nhầm bài.
- **Phân trang đầy đủ & Bảo toàn định danh cổ điển**: Khi `len(references) < referenceCount`, hệ thống tự gọi phân trang `GET /paper/arXiv:<id>/references?limit=1000` đến hết cursor `next`; biểu thức chính quy bảo toàn cả **arXiv ID thế hệ cũ trước năm 2007** (`hep-th/9901001`) và chỉ nhận `publicationDate` chuẩn `YYYY-MM-DD` (không tự suy diễn ngày giả từ năm). Các bài mới chưa kịp lập chỉ mục trên S2 được tự động cứu hộ bằng trích dẫn `.bib`/`.bbl` thực tế được `\cite{}` trong bài (`bib_fallback`).

## 2. Thống kê số lượng trích dẫn trung bình và phân tích nguyên nhân khuyết trường

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
df_ref_kpi = pd.DataFrame([
    ("Bài báo có kết quả S2 xác minh hoàn chỉnh (success + empty)", f"{fmt_int(m1['verified_reference_papers'])} / 25.000", fmt_pct(m1["verified_reference_papers"] / scope["papers"], 2, True)),
    ("Bài báo có dữ liệu trích dẫn hợp lệ (tính cả partial + bib_fallback)", f"{fmt_int(m1['verified_reference_papers'] + m1['reference_status'].get('partial', 0) + m1['reference_status'].get('bib_fallback', 0))} / 25.000", fmt_pct((m1["verified_reference_papers"] + 93 + 14) / scope["papers"], 2, True)),
    ("Số lượng working references trung bình / bài S2 xác minh", f"{fmt_num(m1['mean_references_verified_papers'], 2)} trích dẫn/bài", f"Tổng {fmt_int(m1['working_reference_rows'])} dòng (0 mồ côi)"),
    ("Số lượng trích dẫn có đích arXiv TB / bài (course_export)", f"{fmt_num(m1['arxiv_reference_rows'] / scope['papers'], 2)} cạnh arXiv/bài", f"{fmt_int(m1['arxiv_reference_rows'])} cạnh ({fmt_pct(m1['arxiv_reference_rows']/m1['working_reference_rows'], 2, True)})"),
], columns=["Chỉ tiêu trích dẫn bắt buộc M1", "Kết quả đo", "Tỷ lệ / Quy mô"])
display(style_table(df_ref_kpi, caption="Bảng 3.1. Tổng hợp độ phủ và mật độ trích dẫn trung bình trên 25.000 bài báo."))

rs = pd.Series(m1["reference_status"]).sort_values(ascending=False)
rs_vi = {
    "success": "Thành công đủ mục (success)",
    "empty": "Xác nhận 0 trích dẫn trên S2 (empty)",
    "bib_pending": "Tạm giữ từ .bib/.bbl do S2 429 cuối phiên",
    "api_error": "Lỗi tạm thời API / hết ngân sách retry",
    "partial": "Lệch nhẹ bộ đếm referenceCount (partial)",
    "bib_fallback": "S2 HTTP 404 → cứu hộ từ .bib/.bbl",
    "not_indexed": "S2 HTTP 404 & bài PDF-only (not_indexed)",
}
df_rs = pd.DataFrame({
    "Trạng thái thu thập S2": [rs_vi.get(k, k) for k in rs.index],
    "Mã trạng thái": rs.index,
    "Số bài báo": rs.values,
    "Tỷ lệ (%)": 100.0 * rs.values / scope["papers"],
})
display(style_table(
    df_rs,
    caption="Bảng 3.2. Phân bố trạng thái truy vấn Semantic Scholar trên 25.000 bài báo.",
    int_cols=["Số bài báo"],
    pct_cols=["Tỷ lệ (%)"],
    bar_cols={"Số bài báo": ("var(--ah-bar-fill)", None)},
))

miss_r = pd.Series(m1["references_missing_fraction"])
df_miss_r = pd.DataFrame({
    "Trường bảng working references": miss_r.index,
    "Thu thập thành công (%)": (1.0 - miss_r.values) * 100.0,
    "Tỷ lệ thiếu / null (%)": miss_r.values * 100.0,
})
display(style_table(
    df_miss_r,
    caption="Bảng 3.3. Tỷ lệ thu thập thành công từng trường metadata trên 976.351 dòng trích dẫn.",
    pct_cols=["Thu thập thành công (%)", "Tỷ lệ thiếu / null (%)"],
    bar_cols={"Thu thập thành công (%)": ("rgba(13, 148, 136, 0.22)", 100.0)},
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.1), gridspec_kw={"width_ratios": [1.18, 1.0]})
y1 = np.arange(len(df_rs))
ax1.barh(y1, df_rs["Số bài báo"], color=[BOOK_PALETTE["emerald"], BOOK_PALETTE["teal"], BOOK_PALETTE["amber"], BOOK_PALETTE["crimson"], BOOK_PALETTE["sky"], "#64748B", "#94A3B8"], height=0.58)
ax1.set_yticks(y1); ax1.set_yticklabels(df_rs["Mã trạng thái"]); ax1.invert_yaxis()
ax1.set_xscale("symlog", linthresh=30); ax1.set_xlim(0, 80000); ax1.grid(axis="x")
ax1.set_xlabel("Số bài báo (thang symlog)"); ax1.set_title("Hình 3.1a. Trạng thái S2 theo bài báo")
for i_r, v_r in enumerate(df_rs["Số bài báo"]):
    ax1.text(v_r * 1.16 + 1.5, i_r, fmt_int(v_r), va="center", fontsize=8.1, fontweight="bold")

y2 = np.arange(len(df_miss_r))
ax2.barh(y2, df_miss_r["Thu thập thành công (%)"], color=BOOK_PALETTE["teal"], height=0.55)
ax2.set_yticks(y2); ax2.set_yticklabels(df_miss_r["Trường bảng working references"]); ax2.invert_yaxis()
ax2.set_xlim(0, 116); ax2.grid(axis="x")
ax2.set_xlabel("Tỷ lệ có dữ liệu (%)"); ax2.set_title("Hình 3.1b. Độ đầy đủ trường trên 976.351 dòng")
for i_m, v_m in enumerate(df_miss_r["Thu thập thành công (%)"]):
    ax2.text(v_m + 1.5, i_m, fmt_pct(v_m), va="center", fontsize=8.2, fontweight="bold")
plt.tight_layout(); plt.show()

```{note}
**Giải thích nguyên nhân thực nghiệm trên tập bài mới `2024–2025`:**
- **`3.245` bài `empty` (`12,98%`) & `18` bài S2 HTTP 404**: Do độ trễ bóc tách PDF của Semantic Scholar đối với các bản thảo mới nộp cuối 2024 – đầu 2025 (bản ghi bài báo đã có `referenceCount = 0`, hoặc bài chuyên ngành hẹp chưa kịp ánh xạ mã `arXiv:` → được cứu hộ sang `bib_fallback`).
- **`93` bài `partial` (`0,37%`)**: Phát sinh khi bộ đếm `referenceCount` nội bộ của S2 lớn hơn 1–2 đơn vị so với danh sách thực trả về dù đã phân trang hết `next = null` (do tài liệu tham khảo bị gộp/ẩn phía nhà cung cấp); hệ thống giữ nguyên dữ liệu nhưng gắn nhãn trung thực `partial`.
- **`ref_arxiv_id` đạt `46,72%` (`456.149 / 976.351`)**: Phản ánh đúng cấu trúc học thuật thực tế khi hơn một nửa tài liệu được trích dẫn là sách chuyên khảo, chuẩn kỹ thuật hoặc bài kỷ yếu/tạp chí đóng không phát hành preprint trên arXiv.
```

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/reference_scraper.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_scraper.py).
- Mã nguồn triển khai: [`src/crawler/reference_batch.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_batch.py).
- Mã nguồn triển khai: [`src/crawler/reference_retry.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_retry.py).
- Mã nguồn triển khai: [`src/crawler/enrich_references.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/enrich_references.py).
- Mã nguồn triển khai: [`src/crawler/observations.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/observations.py).